# NASA climate: baselines change the question

How can a line, a warming-stripe panel and a decade summary tell different truths about the same global anomaly estimates? We retain the 1951–1980 reference, and explicitly distinguish smoothing from new observations.

## Objectives

- Validate annual estimates against retained raw J-D values
- Compare unsmoothed, smoothed and rebased views
- Explain how aggregation and color limits affect interpretation

## Start in a fresh kernel

Run every code cell from the top. The Book shows publication-time outputs;
JupyterLite uses the Python (Pyodide) kernel and downloads its runtime and
packages initially. All observations and metadata below come from packaged
snapshots: no live API, token, paid service or GPU is required. If a file is
missing or modified, validation fails explicitly; restore the supplied files.
Download notebook edits and exports to keep them outside browser storage.
The Python code is visible so that transformations can be inspected.

In [ ]:
import sys
from pathlib import Path

if sys.platform == "emscripten":
    import piplite
    await piplite.install(["numpy", "pandas", "matplotlib"])

# Book starts here; desktop/Lite may start at either the notebook or content root.
candidates = [Path.cwd(), Path.cwd() / "repository_examples_v2",
              Path.cwd() / "repository_examples",
              Path.cwd() / "content" / "repository_examples"]
lesson_dir = next((p for p in candidates if (p / "repo_examples.py").is_file()), None)
if lesson_dir is None:
    raise FileNotFoundError("Open this lesson from content/repository_examples on desktop, or repository_examples_v2 in Lite, with repo_examples.py and data/ alongside it.")
sys.path.insert(0, str(lesson_dir))
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from IPython.display import display, HTML
import importlib
import repo_examples as rx
# Pick up a published helper correction when rerunning in an existing kernel.
rx = importlib.reload(rx)

if sys.platform != "emscripten":
    get_ipython().run_line_magic("matplotlib", "inline")

plt.rcParams.update({"figure.figsize": (9, 4.8), "axes.spines.top": False,
                     "axes.spines.right": False, "font.size": 10})
display(rx.validate_all())

## Evidence and attribution

The cards describe the evidence type before any visualization. The manifest retains exact URLs/queries, retrieval UTC, SHA-256 checksums, field order, row counts and transformation recipes. Dataset terms remain separate from the undecided project license.

In [ ]:
display(rx.source_card("nasa"))

In [ ]:
climate = rx.read_csv("nasa/global-temperature.csv")
raw = pd.read_csv(rx.checked_path("nasa/nasa-gistemp-raw.csv"), skiprows=1, na_values="***")
months = ["Jan", "Feb", "Mar", "Apr", "May", "Jun", "Jul", "Aug", "Sep", "Oct", "Nov", "Dec"]
assert raw.loc[raw.Year.eq(2025), months + ["J-D"]].notna().all(axis=None)
assert raw.loc[raw.Year.eq(2026), "J-D"].isna().all()
paired = climate.merge(raw[["Year", "J-D"]], left_on="year", right_on="Year", validate="one_to_one")
assert len(paired) == 146 and np.allclose(paired.anomaly_c, paired["J-D"], equal_nan=True)
assert climate.year.is_unique and climate.year.min() == 1880 and climate.year.max() == 2025
display(climate.iloc[[0, 70, 100, -1]])
display(climate.data_type.value_counts().rename("rows"))

## View 1: annual estimates and an explicit smoother

The thin line retains year-to-year variation; the centered 11-year rolling mean needs a complete window, so it stops five years before each edge. It is a derived summary, not an observed future value. The horizontal zero is the published 1951–1980 reference, not an absence of temperature or risk.

**Method connection:** compare the catalog's Line chart (§5), Area chart (§8) and Table (§3). The line makes ordering and edge gaps immediately visible; the stripe/heatmap panels below extend the catalog with a color-based temporal comparison. A global series provides context; it cannot establish a local health effect.

In [ ]:
smooth = climate.anomaly_c.rolling(11, center=True, min_periods=11).mean()
fig, ax = plt.subplots()
ax.plot(climate.year, climate.anomaly_c, color="#86949a", lw=1, label="Annual published estimate")
ax.plot(climate.year, smooth, color="#b34d35", lw=2.5, label="Centered 11-year mean (derived)")
ax.axhline(0, color="black", lw=.7)
rx.finish(ax, "Global land–ocean temperature anomaly", "Year", "Anomaly (°C), reference 1951–1980")
ax.legend()
fig.tight_layout()
plt.show()

## View 2: fixed color limits and a heatmap

The stripe panel removes the y axis and emphasizes the sequence of signs and magnitudes. Its shared symmetric color limits make zero meaningful. The decade-by-year grid exposes the final incomplete decade rather than filling it with invented values. Compare the exact numeric table when color differences become hard to discriminate.

In [ ]:
climate["decade"] = climate.year // 10 * 10
climate["year_in_decade"] = climate.year % 10
grid = climate.pivot(index="decade", columns="year_in_decade", values="anomaly_c")
limit = float(np.ceil(climate.anomaly_c.abs().max()*10)/10)
fig, axes = plt.subplots(2, 1, figsize=(11, 6), gridspec_kw={"height_ratios": [1, 4]}, layout="constrained")
stripe = axes[0].imshow(climate.anomaly_c.to_numpy()[None, :], aspect="auto", cmap="RdBu_r", vmin=-limit, vmax=limit,
                        extent=[1879.5, 2025.5, 0, 1])
axes[0].set(yticks=[], xlabel="Year", title="Annual estimates — fixed diverging scale")
axes[1].imshow(np.ma.masked_invalid(grid.to_numpy()), aspect="auto", cmap="RdBu_r", vmin=-limit, vmax=limit)
axes[1].set(xticks=range(10), xlabel="Year within decade", yticks=range(len(grid)), yticklabels=grid.index,
            ylabel="Decade starts", title="Empty cells are missing years, not zero anomaly")
fig.colorbar(stripe, ax=axes, label="°C relative to 1951–1980", shrink=.85)
plt.show()

## View 3: rebase without changing year-to-year differences

Subtracting the mean for a different baseline shifts every value by a constant. The difference between 2000 and 2020 remains identical. The newer-baseline plot answers a changed reference question; it does not undo the estimated warming. Values may not average exactly zero over the published baseline because the product uses its own gridding, weighting and rounding procedures.

In [ ]:
baseline = climate.loc[climate.year.between(1991, 2020), "anomaly_c"].mean()
climate["relative_1991_2020_c"] = climate.anomaly_c - baseline
assert np.allclose(np.diff(climate.anomaly_c), np.diff(climate.relative_1991_2020_c))
decades = climate.groupby("decade").agg(mean_c=("anomaly_c", "mean"), years=("year", "size"))
display(decades.tail(6))
fig, ax = plt.subplots()
ax.plot(climate.year, climate.anomaly_c, label="Published reference 1951–1980")
ax.plot(climate.year, climate.relative_1991_2020_c, label="Rebased to arithmetic 1991–2020 mean")
ax.axhline(0, color="black", lw=.7)
rx.finish(ax, "Rebasing changes the origin", "Year", "Anomaly (°C; reference in legend)")
ax.legend()
fig.tight_layout()
plt.show()
print(f"Rebasing constant: {baseline:.3f} °C")

## Exercise

Change the smoothing window to 21 years. Count and explain the added missing edge values. Then summarize complete decades only: show how excluding the six-year 2020s block changes the comparison without modifying any annual estimate. Describe the pattern with a stated baseline and avoid local health claims.

## Limitations and reflection

This product is an observational global estimate with methodological uncertainty; this extract contains point estimates without uncertainty bounds. The retained raw table has all twelve monthly values and J-D for 2025. Its incomplete 2026 row has missing J-D and is excluded by the original recipe; the tidy annual series therefore ends in 2025. Rounding, revisions and different baselines matter. No local temperature, exposure or disease observations are included.

## Citations and repository connections

[NASA GISTEMP sources and citations](https://data.giss.nasa.gov/gistemp/); GISTEMP Team (2026); Lenssen et al. (2024), [doi:10.1029/2023JD040179](https://doi.org/10.1029/2023JD040179). The files are retained from [Climate-and-Health at 248f9860d189](https://github.com/jltobias/JupyterLite-Climate-and-Health/tree/248f9860d189502013b36b3c4271777edafc4f7f/content/data), with its original retrieval/provenance. This lesson connects that repository's evidence registry to line, heatmap and small-multiple methods.

See the [companion index](index.md) and [full attribution](https://jltobias.github.io/JupyterLite-Data-Visualization/book/references.html). These are original teaching code and narrative; linked repository notebooks and authored figures have not been copied.